## inventaire_fichiers_bofip

**Fichier(s) source :** `./data/bofip_stock_live_20260521.tgz` (stock du 21.05.2026)

**Fichier(s) de sortie :** DataFrame en mémoire

**Description :** Inventaire des fichiers de l'archive BOFiP : nombre de fichiers et d'objets, répartition par famille, et liens entre Contenu et les autres familles (avec renvois orphelins).

## Étape 1 — Lecture complète de l'archive

On ouvre le .tgz une seule fois et on stocke la liste de tous les fichiers en mémoire. Cette cellule prend environ 30 secondes.

In [1]:
import tarfile
import os
from collections import Counter, defaultdict
import pandas as pd

ARCHIVE = r"./data/bofip_stock_live_20260521.tgz"

if not os.path.isfile(ARCHIVE):
    raise FileNotFoundError(f"Fichier introuvable : {ARCHIVE}")

print(f"Archive : {os.path.basename(ARCHIVE)}")
print(f"Taille : {os.path.getsize(ARCHIVE) / (1024*1024):.1f} Mo")
print("Lecture en cours...")

tous_fichiers = []   # liste de tous les chemins de fichiers
tous_dossiers = []   # liste de tous les chemins de dossiers

with tarfile.open(ARCHIVE, "r:gz") as tar:
    for m in tar.getmembers():
        if m.isfile():
            tous_fichiers.append(m.name)
        elif m.isdir():
            tous_dossiers.append(m.name)

print(f"\nLecture terminée.")
print(f"Fichiers : {len(tous_fichiers):,}")
print(f"Dossiers : {len(tous_dossiers):,}")

Archive : bofip_stock_live_20260521.tgz
Taille : 111.0 Mo
Lecture en cours...

Lecture terminée.
Fichiers : 14,993
Dossiers : 15,095


## Étape 2 — Noms de fichiers les plus fréquents

In [2]:
basenames = [os.path.basename(f) for f in tous_fichiers]

print("Les 10 noms de fichiers les plus fréquents :")
for nom, n in Counter(basenames).most_common(10):
    print(f"  {nom:25s} : {n:,}")

Les 10 noms de fichiers les plus fréquents :
  document.xml              : 7,495
  data.html                 : 6,311
  data1.jpg                 : 532
  data1.JPG                 : 190
  data1.png                 : 153
  data1.pdf                 : 112
  data1                     : 81
  data1.PNG                 : 63
  data1.jpeg                : 27
  data1.odt                 : 10


## Étape 3 — Identification des objets et des familles

Un objet est un dossier qui contient un fichier `document.xml`. La famille se lit dans le chemin.

In [3]:
# Regrouper les fichiers par dossier parent
par_dossier = defaultdict(list)
for f in tous_fichiers:
    dossier = os.path.dirname(f).replace("\\", "/")
    par_dossier[dossier].append(os.path.basename(f))

# Garder seulement les dossiers qui contiennent un document.xml
objets = []
for dossier, fichiers in par_dossier.items():
    if "document.xml" not in fichiers:
        continue
    
    parts = dossier.split("/")
    identifiant = parts[-1]
    
    # Détecter la famille
    d_lower = dossier.lower()
    if "contenu" in d_lower:
        famille = "Contenu"
    elif "image" in d_lower:
        famille = "Image"
    elif "attachment" in d_lower or "fichier" in d_lower:
        famille = "Attachment"
    elif "plan" in d_lower:
        famille = "PlanClassement"
    else:
        famille = "Autre"
    
    objets.append({
        "identifiant": identifiant,
        "famille": famille,
        "chemin": dossier,
        "nb_fichiers": len(fichiers),
        "a_html": "data.html" in fichiers,
        "a_pdf": any(f.endswith(".pdf") for f in fichiers),
        "liste_fichiers": ", ".join(sorted(fichiers))
    })

df = pd.DataFrame(objets)
print(f"Total objets : {len(df):,}")
print()
for fam, n in df["famille"].value_counts().items():
    print(f"  {fam:20s} : {n:,} objets")

Total objets : 7,495

  Contenu              : 6,311 objets
  Image                : 1,041 objets
  Attachment           : 142 objets
  PlanClassement       : 1 objets


## Étape 4 — Tableau récapitulatif : objets vs. fichiers

In [4]:
recap = df.groupby("famille").agg(
    objets=("identifiant", "count"),
    fichiers=("nb_fichiers", "sum"),
    avec_html=("a_html", "sum"),
    avec_pdf=("a_pdf", "sum")
).reset_index()
recap = recap.sort_values("objets", ascending=False)
recap["ratio"] = (recap["fichiers"] / recap["objets"]).round(1)

# Ligne total
total_row = pd.DataFrame([{
    "famille": "TOTAL",
    "objets": recap["objets"].sum(),
    "fichiers": recap["fichiers"].sum(),
    "avec_html": int(recap["avec_html"].sum()),
    "avec_pdf": int(recap["avec_pdf"].sum()),
    "ratio": ""
}])
recap_affich = pd.concat([recap, total_row], ignore_index=True)

print("=" * 75)
print("TABLEAU — Objets et fichiers par famille")
print("=" * 75)
print(recap_affich.to_string(index=False))
print("=" * 75)

TABLEAU — Objets et fichiers par famille
       famille  objets  fichiers  avec_html  avec_pdf ratio
       Contenu    6311     12622       6311         0   2.0
         Image    1041      2082          0         0   2.0
    Attachment     142       284          0       112   2.0
PlanClassement       1         2          0         0   2.0
         TOTAL    7495     14990       6311       112      


## Étape 5 — Exemples concrets par famille

In [5]:
for fam in ["Contenu", "Image", "Attachment", "PlanClassement"]:
    sous = df[df["famille"] == fam].head(3)
    if len(sous) > 0:
        print(f"\n--- {fam} (3 premiers) ---")
        for _, r in sous.iterrows():
            print(f"  {r['identifiant']:15s} | {r['nb_fichiers']} fichiers | {r['liste_fichiers']}")


--- Contenu (3 premiers) ---
  2023-01-18      | 2 fichiers | data.html, document.xml
  2023-01-18      | 2 fichiers | data.html, document.xml
  2022-03-23      | 2 fichiers | data.html, document.xml

--- Image (3 premiers) ---
  2020-04-15      | 2 fichiers | data1.JPG, document.xml
  2016-03-02      | 2 fichiers | data1.JPG, document.xml
  2016-03-02      | 2 fichiers | data1.JPG, document.xml

--- Attachment (3 premiers) ---
  2015-05-06      | 2 fichiers | data1.pdf, document.xml
  2015-07-01      | 2 fichiers | data1.pdf, document.xml
  2015-08-05      | 2 fichiers | data1.pdf, document.xml

--- PlanClassement (3 premiers) ---
  2026-05-21      | 2 fichiers | data.xml, document.xml


## Étape 6 — Liens entre Contenu et les autres familles

On ouvre l'archive une seconde fois pour lire les `document.xml` des Contenu et extraire les `dc:relation`. Cette cellule prend environ 1 à 2 minutes.

In [6]:
import xml.etree.ElementTree as ET

# Construire l'ensemble des chemins document.xml des Contenu
chemins_contenu_xml = set()
for _, r in df[df["famille"] == "Contenu"].iterrows():
    chemins_contenu_xml.add(r["chemin"] + "/document.xml")

print(f"Document.xml de Contenu à lire : {len(chemins_contenu_xml):,}")
print("Lecture en cours (1-2 minutes)...")

liens = []
erreurs = 0
compteur = 0

with tarfile.open(ARCHIVE, "r:gz") as tar:
    for m in tar.getmembers():
        if not m.isfile():
            continue
        chemin = m.name.replace("\\", "/")
        if chemin not in chemins_contenu_xml:
            continue
        
        parts = chemin.split("/")
        source_id = parts[-2] if len(parts) >= 2 else "inconnu"
        
        try:
            f = tar.extractfile(m)
            if f is None:
                continue
            tree = ET.parse(f)
            root = tree.getroot()
            
            for elem in root.iter():
                tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
                if tag == "relation" and elem.text:
                    texte = elem.text.strip()
                    # Extraire le type de relation
                    rel_type = ""
                    for k, v in elem.attrib.items():
                        if "type" in k.lower():
                            rel_type = v
                            break
                    
                    if ":" in texte:
                        fam_cible, id_cible = texte.split(":", 1)
                    else:
                        fam_cible, id_cible = "Inconnu", texte
                    
                    liens.append({
                        "source": source_id,
                        "type_relation": rel_type,
                        "cible_famille": fam_cible,
                        "cible_id": id_cible
                    })
            compteur += 1
            if compteur % 1000 == 0:
                print(f"  {compteur:,} documents lus...")
        except Exception:
            erreurs += 1

df_liens = pd.DataFrame(liens)
print(f"\nTerminé. {compteur:,} documents lus, {len(df_liens):,} liens extraits.")
if erreurs > 0:
    print(f"Erreurs : {erreurs}")

Document.xml de Contenu à lire : 6,311
Lecture en cours (1-2 minutes)...
  1,000 documents lus...
  2,000 documents lus...
  3,000 documents lus...
  4,000 documents lus...
  5,000 documents lus...
  6,000 documents lus...

Terminé. 6,311 documents lus, 23,128 liens extraits.


## Étape 7 — Tableau croisé : qui pointe vers quoi ?

In [7]:
if len(df_liens) > 0:
    print("Liens par famille cible :")
    print(df_liens["cible_famille"].value_counts().to_string())
    print()
    print("Liens par type de relation :")
    print(df_liens["type_relation"].value_counts().to_string())
    print()
    print("Tableau croisé : famille cible × type de relation")
    print(pd.crosstab(df_liens["cible_famille"], df_liens["type_relation"], margins=True).to_string())
else:
    print("Aucun lien extrait.")

Liens par famille cible :
cible_famille
Contenu      21403
Actualite     1072
Image          536
Fichier        117

Liens par type de relation :
type_relation
references    22475
requires        653

Tableau croisé : famille cible × type de relation
type_relation  references  requires    All
cible_famille                             
Actualite            1072         0   1072
Contenu             21403         0  21403
Fichier                 0       117    117
Image                   0       536    536
All                 22475       653  23128


## Étape 8 — Orphelins

In [8]:
if len(df_liens) > 0:
    ids_presents = set(df["identifiant"].values)
    df_liens["cible_presente"] = df_liens["cible_id"].isin(ids_presents)
    
    orphelins = df_liens[~df_liens["cible_presente"]]
    
    print(f"Liens résolus : {df_liens['cible_presente'].sum():,}")
    print(f"Liens orphelins : {len(orphelins):,} ({len(orphelins)/len(df_liens)*100:.1f} %)")
    print()
    print("Orphelins par famille cible :")
    print(orphelins["cible_famille"].value_counts().to_string())
    print()
    print("Documents Contenu reliés à d'autres familles :")
    for fam in df_liens["cible_famille"].unique():
        n = df_liens[df_liens["cible_famille"] == fam]["source"].nunique()
        print(f"  vers {fam:15s} : {n:,} documents")
else:
    print("Pas de liens.")

Liens résolus : 0
Liens orphelins : 23,128 (100.0 %)

Orphelins par famille cible :
cible_famille
Contenu      21403
Actualite     1072
Image          536
Fichier        117

Documents Contenu reliés à d'autres familles :
  vers Contenu         : 695 documents
  vers Actualite       : 277 documents
  vers Image           : 89 documents
  vers Fichier         : 33 documents


## Étape 9 — Synthèse

In [9]:
print("=" * 70)
print("SYNTHÈSE")
print("=" * 70)
print(f"Archive : {os.path.basename(ARCHIVE)}")
print(f"Fichiers dans l'archive : {len(tous_fichiers):,}")
print(f"Objets (avec document.xml) : {len(df):,}")
print(f"Ratio : {len(tous_fichiers)/len(df):.1f} fichiers par objet")
print()
for fam in ["Contenu", "Image", "Attachment", "PlanClassement"]:
    n = len(df[df["famille"] == fam])
    print(f"  {fam:20s} : {n:>6,} objets")
print(f"  {'TOTAL':20s} : {len(df):>6,} objets")
print()
print(f"Conclusion pour le chapitre :")
print(f"  {len(tous_fichiers):,} fichiers dans l'archive")
print(f"  = {len(df):,} objets (chaque objet = document.xml + contenu)")
print(f"  = {len(df[df['famille']=='Contenu']):,} Contenu (doctrine, 6 types) + ressources liées")
print("=" * 70)

SYNTHÈSE
Archive : bofip_stock_live_20260521.tgz
Fichiers dans l'archive : 14,993
Objets (avec document.xml) : 7,495
Ratio : 2.0 fichiers par objet

  Contenu              :  6,311 objets
  Image                :  1,041 objets
  Attachment           :    142 objets
  PlanClassement       :      1 objets
  TOTAL                :  7,495 objets

Conclusion pour le chapitre :
  14,993 fichiers dans l'archive
  = 7,495 objets (chaque objet = document.xml + contenu)
  = 6,311 Contenu (doctrine, 6 types) + ressources liées
